## Imports

In [25]:
import re
import json
from pathlib import Path
from collections import defaultdict
import pandas as pd

## Load Raw Data

In [26]:
raw_data_dir = Path("../data/raw")
languages = ["english", "tagalog", "cebuano", "waray"]

# Dictionary to store raw data: {language: {(book, chapter, verse): text}}
raw_data = {}

for lang in languages:
    file_path = raw_data_dir / f"{lang}_raw.txt"
    raw_data[lang] = {}
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            # Parse format: BOOK CHAPTER:VERSE TEXT
            match = re.match(r'^(\w+)\s+(\d+):(\d+)\s+(.+)$', line)
            if match:
                book, chapter, verse, text = match.groups()
                key = (book, int(chapter), int(verse))
                raw_data[lang][key] = text
    
    print(f"Loaded {lang}: {len(raw_data[lang])} verses")

print(f"\nTotal languages loaded: {len(raw_data)}")

Loaded english: 29147 verses
Loaded tagalog: 16238 verses
Loaded cebuano: 29094 verses
Loaded waray: 28223 verses

Total languages loaded: 4


## Data Cleaning Functions

Define functions to clean and normalize the text data.

In [27]:
def clean_text(text):
    """
    Clean and normalize text:
    - Remove leading verse numbers (e.g., "1 The book..." -> "The book...")
    - Normalize whitespace
    - Remove extra spaces
    """
    # Remove leading verse numbers at the beginning of text
    text = re.sub(r'^\d+\s+', '', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

def is_valid_verse(text):
    """
    Check if a verse is valid (not empty, not marked as MISSING)
    """
    if not text or text.strip() == "":
        return False
    if "MISSING" in text:
        return False
    return True

## Clean and Process Data

Apply cleaning functions to all verses and organize by language.

In [28]:
# Clean and process all verses
cleaned_data = {}
stats = {lang: {"total": 0, "valid": 0, "invalid": 0} for lang in languages}

for lang in languages:
    cleaned_data[lang] = {}
    
    for key, text in raw_data[lang].items():
        cleaned_text = clean_text(text)
        
        stats[lang]["total"] += 1
        if is_valid_verse(cleaned_text):
            cleaned_data[lang][key] = cleaned_text
            stats[lang]["valid"] += 1
        else:
            stats[lang]["invalid"] += 1

# Display statistics
print("Cleaning Statistics:")
print("=" * 60)
for lang in languages:
    s = stats[lang]
    print(f"{lang:12} | Total: {s['total']:4} | Valid: {s['valid']:4} | Invalid: {s['invalid']:3}")
print("=" * 60)

Cleaning Statistics:
english      | Total: 29147 | Valid: 29147 | Invalid:   0
tagalog      | Total: 16238 | Valid: 16238 | Invalid:   0
cebuano      | Total: 29094 | Valid: 29094 | Invalid:   0
waray        | Total: 28223 | Valid: 28223 | Invalid:   0


## Identify Common Verses Across Languages

In [29]:
# Find verses that exist in all languages
all_verse_keys = [set(cleaned_data[lang].keys()) for lang in languages]
common_verses = set.intersection(*all_verse_keys)

print(f"Total common verses across all {len(languages)} languages: {len(common_verses)}")

# Sort common verses by book, chapter, verse
common_verses_sorted = sorted(list(common_verses))

# Display breakdown by book
books_count = defaultdict(int)
for book, chapter, verse in common_verses_sorted:
    books_count[book] += 1

print("\nBreakdown by book:")
for book in sorted(books_count.keys()):
    print(f"  {book}: {books_count[book]} verses")

Total common verses across all 4 languages: 15783

Breakdown by book:
  1JN: 105 verses
  1PE: 105 verses
  1SA: 803 verses
  1TH: 89 verses
  1TI: 112 verses
  2CO: 255 verses
  2JN: 13 verses
  2PE: 61 verses
  2SA: 675 verses
  2TH: 46 verses
  2TI: 83 verses
  3JN: 15 verses
  ACT: 998 verses
  COL: 95 verses
  DEU: 946 verses
  EPH: 155 verses
  EXO: 1199 verses
  GAL: 149 verses
  GEN: 1502 verses
  HEB: 303 verses
  JAS: 107 verses
  JDG: 606 verses
  JHN: 873 verses
  JOS: 645 verses
  JUD: 25 verses
  LEV: 823 verses
  LUK: 1145 verses
  MAT: 1053 verses
  MRK: 672 verses
  NUM: 1038 verses
  PHM: 25 verses
  PHP: 104 verses
  REV: 402 verses
  ROM: 431 verses
  RUT: 79 verses
  TIT: 46 verses


## Create Segmented Data Structure

Organize cleaned verses by language with their references for easy parallel corpus creation.

In [30]:
# Create a structured dataset for each language
# Format: list of dicts with keys: book, chapter, verse, text, reference
segmented_data = {}

for lang in languages:
    segmented_data[lang] = []
    
    for key in common_verses_sorted:
        book, chapter, verse = key
        text = cleaned_data[lang][key]
        
        segmented_data[lang].append({
            "book": book,
            "chapter": chapter,
            "verse": verse,
            "reference": f"{book} {chapter}:{verse}",
            "text": text
        })

# Display sample from each language
print("Sample verses from each language:")
print("=" * 80)
for lang in languages:
    print(f"\n{lang.upper()}:")
    sample = segmented_data[lang][0]  # First verse
    print(f"  Reference: {sample['reference']}")
    print(f"  Text: {sample['text'][:100]}...")
print("=" * 80)

Sample verses from each language:

ENGLISH:
  Reference: 1JN 1:1
  Text: That which was from the beginning, which we have heard, which we have seen with our eyes, which we h...

TAGALOG:
  Reference: 1JN 1:1
  Text: Yaong buhat sa pasimula, na aming narinig, nakita ng aming mga mata, aming napagmasdan, at nahipo ng...

CEBUANO:
  Reference: 1JN 1:1
  Text: Kadto nga diha na sukad pa sa sinugdan nga among nadungog, among nakita pinaagi sa among mga mata, a...

WARAY:
  Reference: 1JN 1:1
  Text: Nagsusurat kami ha iyo mahitungod han Pulong nga nahatag han kinabuhi nga nakada na nga daan tikang ...


## Save Processed Data

Save the cleaned and segmented data in multiple formats for easy use in the next notebook.

In [31]:
# Create processed directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save each language as JSON (with metadata)
for lang in languages:
    output_file = processed_dir / f"{lang}_clean.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(segmented_data[lang], f, ensure_ascii=False, indent=2)
    print(f"Saved {lang} to {output_file}")

# Save each language as simple text file (one verse per line)
for lang in languages:
    output_file = processed_dir / f"{lang}_clean.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        for verse in segmented_data[lang]:
            f.write(f"{verse['text']}\n")
    print(f"Saved {lang} to {output_file}")

# Save common verse references
references_file = processed_dir / "verse_references.txt"
with open(references_file, "w", encoding="utf-8") as f:
    for book, chapter, verse in common_verses_sorted:
        f.write(f"{book} {chapter}:{verse}\n")
print(f"\nSaved verse references to {references_file}")

print(f"\n✓ All processed data saved to {processed_dir}")

Saved english to ..\data\processed\english_clean.json
Saved tagalog to ..\data\processed\tagalog_clean.json
Saved cebuano to ..\data\processed\cebuano_clean.json
Saved waray to ..\data\processed\waray_clean.json
Saved english to ..\data\processed\english_clean.txt
Saved tagalog to ..\data\processed\tagalog_clean.txt
Saved cebuano to ..\data\processed\cebuano_clean.txt
Saved waray to ..\data\processed\waray_clean.txt

Saved verse references to ..\data\processed\verse_references.txt

✓ All processed data saved to ..\data\processed


## Summary Statistics

Display final statistics about the cleaned and processed data.

In [32]:
# Calculate statistics
print("\n" + "=" * 80)
print("PREPROCESSING SUMMARY")
print("=" * 80)

print(f"\nTotal languages: {len(languages)}")
print(f"Languages: {', '.join(languages)}")

print(f"\nCommon verses across all languages: {len(common_verses_sorted)}")

print("\nWord count statistics:")
for lang in languages:
    total_words = sum(len(verse['text'].split()) for verse in segmented_data[lang])
    avg_words = total_words / len(segmented_data[lang]) if segmented_data[lang] else 0
    print(f"  {lang:12}: {total_words:6,} words | Avg per verse: {avg_words:.1f}")

print("\nBooks included:")
for book in sorted(books_count.keys()):
    print(f"  {book}: {books_count[book]} verses")

print("\n" + "=" * 80)
print("✓ Data is ready for parallel corpus creation!")
print("=" * 80)


PREPROCESSING SUMMARY

Total languages: 4
Languages: english, tagalog, cebuano, waray

Common verses across all languages: 15783

Word count statistics:
  english     : 374,717 words | Avg per verse: 23.7
  tagalog     : 406,286 words | Avg per verse: 25.7
  cebuano     : 420,974 words | Avg per verse: 26.7
  waray       : 423,241 words | Avg per verse: 26.8

Books included:
  1JN: 105 verses
  1PE: 105 verses
  1SA: 803 verses
  1TH: 89 verses
  1TI: 112 verses
  2CO: 255 verses
  2JN: 13 verses
  2PE: 61 verses
  2SA: 675 verses
  2TH: 46 verses
  2TI: 83 verses
  3JN: 15 verses
  ACT: 998 verses
  COL: 95 verses
  DEU: 946 verses
  EPH: 155 verses
  EXO: 1199 verses
  GAL: 149 verses
  GEN: 1502 verses
  HEB: 303 verses
  JAS: 107 verses
  JDG: 606 verses
  JHN: 873 verses
  JOS: 645 verses
  JUD: 25 verses
  LEV: 823 verses
  LUK: 1145 verses
  MAT: 1053 verses
  MRK: 672 verses
  NUM: 1038 verses
  PHM: 25 verses
  PHP: 104 verses
  REV: 402 verses
  ROM: 431 verses
  RUT: 79 ver